# U2 AquaVigía v3: inferencia sin estado, error en vivo y observabilidad completa

Dimensión U2 de **Jhenderson Aaron Machaca Mamani**: pH y turbidez. Evento de negocio: **alerta de calidad** (`alerta_calidad`) en `alerta-eventos`.

Esta versión reemplaza a la v2 para cubrir todo el temario de la evaluación S12:

| Tema | Qué agrega este cuaderno |
|---|---|
| S6 | Contrato de entrada **versión 2** (`version_esquema`) y evento de negocio versionado con `evento_id` idempotente |
| S8 | **Deduplicación** dentro del watermark antes de pronosticar (semántica al menos una vez) |
| S9 | **Lag del consumidor**, throughput, disponibilidad y umbrales en Grafana |
| S10 | **Inferencia sin estado**: cada evento trae su contexto, una sola lectura basta |
| S11 | **Error de predicción medido en vivo** y **panel batch** (validación del 12b) junto al de streaming |

Requisito: el tópico `agua-usgs-eventos-v2` (3 particiones) con eventos versión 2, que llegan por MQTT y el puente (`usgs_productor.py --salida mqtt`).

**Episodio de la demostración.** Se reproduce un tramo real del periodo de prueba del modelo (nunca visto al entrenar): del 16-oct-2026 22:00 al 17-oct-2026 04:00 UTC. En ese tramo el oxígeno de USGS-02074556 bajó hacia su límite inferior y varias estaciones salieron de su rango de pH y turbidez. Se usa el umbral de operación (0,5), sin ajustes para la demo.

In [13]:
import os, json, time, math, shutil, uuid, urllib.request, urllib.error
import pandas as pd
from datetime import datetime, timezone
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StringType, DoubleType, IntegerType, MapType
from pyspark.ml import PipelineModel
from pyspark.sql.streaming import StreamingQueryListener

INTEGRANTE = "Jhenderson Machaca"
ID = "jhenderson"
VARIABLES = ["ph", "turbidez_fnu"]
TOPIC_ENTRADA = "agua-usgs-eventos-v2"
TOPIC_NEGOCIO = "alerta-eventos"
TIPO_NEGOCIO = "alerta_calidad"
BROKER = "kafka:9092"
PUSHGATEWAY = "http://pushgateway:9091"
BASE = "/opt/s12-u2-usgs"
ARTIFACTS = f"{BASE}/artifacts"
RUTA_MODELOS = f"{ARTIFACTS}/modelos_{ID}"

MODO = "reproduccion"     # "reproduccion": desde el inicio del topico (earliest). "vivo": solo lo nuevo (latest)
DURACION_MIN = 8          # cuanto corre la demostracion
TRIGGER = "10 seconds"    # cada cuanto se dispara un micro-lote
MAX_POR_LOTE = 600        # mensajes de Kafka por micro-lote: el lag baja de a poco y se ve en Grafana
RIESGO_ALERTA = 0.5       # probabilidad de salir del rango desde la cual se emite el evento de negocio
ENFRIAMIENTO_MIN = 60     # regla de negocio: no repetir el evento para la misma estacion y variable antes de 60 min
WATERMARK = "85 minutes"  # medido en el 12a: p99 del retraso x 1.2
LIMPIAR_CHECKPOINT = True

DIRECCION = {"temperatura_c": "alta", "oxigeno_mgl": "baja", "ph": "ambas", "turbidez_fnu": "alta"}

spark = (SparkSession.builder.appName(f"u2-usgs-negocio-v3-{ID}").master("local[*]")
         .config("spark.jars.packages", "org.apache.spark:spark-sql-kafka-0-10_2.13:4.2.0")
         .config("spark.sql.shuffle.partitions", "1").config("spark.sql.session.timeZone", "America/Lima")
         .config("spark.ui.port", "4041").getOrCreate())
spark.sparkContext.setLogLevel("ERROR")

META = json.load(open(f"{RUTA_MODELOS}/meta.json", encoding="utf-8"))
MODELOS = {v: PipelineModel.load(f"{RUTA_MODELOS}/modelo_{v}_1h") for v in VARIABLES if v in META["variables"]}
print("Modelos cargados (entrenados en el 12b, aqui NO se entrena nada):", list(MODELOS))
for v in MODELOS:
    m = META["variables"][v]
    print(f"  {v}: {m['ganador']}  RMSE={m['rmse']:.4f}  persistencia={m['rmse_persistencia']:.4f}  "
          f"mejora={m['mejora_vs_persistencia_pct']:+.1f} %  features={m['features']}")

Modelos cargados (entrenados en el 12b, aqui NO se entrena nada): ['ph', 'turbidez_fnu']
  ph: LinearRegression  RMSE=0.0683  persistencia=0.0753  mejora=+9.4 %  features=['ph', 'ph_l1', 'ph_l2', 'ph_l3', 'ph_l6', 'ph_l12', 'delta_1h', 'turbidez_fnu', 'hora_sin', 'hora_cos']
  turbidez_fnu: LinearRegression  RMSE=8.2011  persistencia=8.5651  mejora=+4.2 %  features=['turbidez_fnu', 'turbidez_fnu_l1', 'turbidez_fnu_l2', 'turbidez_fnu_l3', 'turbidez_fnu_l6', 'turbidez_fnu_l12', 'delta_1h', 'ph', 'hora_sin', 'hora_cos']


## 1. Contrato de entrada versión 2 y validación

El evento IoT versión 2 es el de la versión 1 más el campo `contexto`:

| Campo | Contenido |
|---|---|
| `version_esquema` | 2 (los eventos viejos no lo traen y se leen como versión 1) |
| `contexto.rezagos` | valores de la misma variable 5, 10, 15, 30 y 60 min antes (`l1`, `l2`, `l3`, `l6`, `l12`) |
| `contexto.simultaneas` | valores de las otras variables de la estación en el mismo bloque de 5 min |

Lo arma el productor, que ya consulta las últimas 3 horas de cada estación en cada ronda. Cada lectura se valida contra el esquema y contra el **rango físico** del sensor.

In [14]:

VARS_CONOCIDAS = ["ph", "oxigeno_mgl", "turbidez_fnu", "conductividad_uscm", "temperatura_c", "nitrato_mgl"]
# Rango FISICO posible del sensor (no el rango "normal" del rio): fuera de esto la lectura es un error del equipo.
RANGO_FISICO = {"temperatura_c": (-5, 45), "oxigeno_mgl": (0, 25), "ph": (0, 14),
                "turbidez_fnu": (0, 4000), "conductividad_uscm": (0, 100000), "nitrato_mgl": (0, 100)}

esquema_v2 = (StructType()
    .add("version_esquema", IntegerType())
    .add("evento_id", StringType()).add("estacion_id", StringType()).add("variable", StringType())
    .add("codigo_parametro", StringType()).add("valor", DoubleType())
    .add("fecha_medicion_utc", StringType()).add("fecha_ingesta_utc", StringType())
    .add("retraso_min", DoubleType()).add("origen", StringType()).add("fuente", StringType())
    .add("contexto", StructType()
         .add("rezagos", MapType(StringType(), DoubleType()))
         .add("simultaneas", MapType(StringType(), DoubleType()))))

def fuente_batch(topic=None):
    return (spark.read.format("kafka").option("kafka.bootstrap.servers", BROKER)
            .option("subscribe", topic or TOPIC_ENTRADA).option("startingOffsets", "earliest")
            .option("includeHeaders", "true").load())

def fuente_stream(inicio="latest", max_por_lote=None):
    r = (spark.readStream.format("kafka").option("kafka.bootstrap.servers", BROKER)
         .option("subscribe", TOPIC_ENTRADA).option("startingOffsets", inicio)
         .option("includeHeaders", "true").option("failOnDataLoss", "false"))
    if max_por_lote:
        r = r.option("maxOffsetsPerTrigger", max_por_lote)
    return r.load()

def publicar_kafka(registros, topic=None):
    """registros: lista de (key, value_json)."""
    (spark.createDataFrame(registros, ["key", "value"]).write.format("kafka")
     .option("kafka.bootstrap.servers", BROKER).option("topic", topic or TOPIC_ENTRADA).save())

def parsear(df):
    """Kafka -> columnas tipadas + estado de validacion. No descarta nada: marca."""
    rango = F.lit(False)
    for v, (lo, hi) in RANGO_FISICO.items():
        rango = F.when(F.col("variable") == v, (F.col("valor") >= lo) & (F.col("valor") <= hi)).otherwise(rango)
    d = (df.select(F.col("key").cast("string").alias("key"), "partition", "offset",
                   F.expr("cast(get(filter(headers, h -> h.key = 'canal'), 0).value as string)").alias("canal"),
                   F.expr("cast(get(filter(headers, h -> h.key = 'mqtt_topic'), 0).value as string)").alias("mqtt_topic"),
                   F.col("value").cast("string").alias("json"))
           .withColumn("e", F.from_json("json", esquema_v2)).select("key", "partition", "offset", "canal", "mqtt_topic", "json", "e.*")
           .withColumn("ts", F.to_timestamp("fecha_medicion_utc")))
    esquema_ok = (F.col("estacion_id").isNotNull() & F.col("variable").isNotNull()
                  & F.col("valor").isNotNull() & F.col("ts").isNotNull())
    return d.withColumn("validacion",
        F.when(~esquema_ok, "esquema_invalido")
         .when(~F.col("variable").isin(VARS_CONOCIDAS), "variable_desconocida")
         .when(~rango, "fuera_de_rango_fisico").otherwise("valido"))

ent = parsear(fuente_batch())
mias = ent.where(F.col("variable").isin(VARIABLES))
print("Mensajes en el topico:", ent.count())
ent.groupBy("version_esquema", "origen", "canal").count().orderBy("version_esquema", "origen").show()
ent.groupBy("validacion").count().show()
completo = F.size("contexto.rezagos") == 5
mias.groupBy("variable").agg(F.count("*").alias("lecturas"),
                             F.sum(completo.cast("int")).alias("con_5_rezagos")).show()

Mensajes en el topico: 44472


+---------------+------+-----+-----+
|version_esquema|origen|canal|count|
+---------------+------+-----+-----+
|           NULL|  NULL| NULL|    4|
|              2|  NULL| NULL|    8|
|              2|replay| mqtt| 8566|
|              2|replay| NULL|  100|
|              2|  vivo| mqtt|35494|
|              2|  vivo| NULL|  300|
+---------------+------+-----+-----+



+--------------------+-----+
|          validacion|count|
+--------------------+-----+
|              valido|44460|
|fuera_de_rango_fi...|    4|
|    esquema_invalido|    8|
+--------------------+-----+



[Stage 136:===================>                                     (1 + 2) / 3]

+------------+--------+-------------+
|    variable|lecturas|con_5_rezagos|
+------------+--------+-------------+
|          ph|    9157|         8022|
|turbidez_fnu|    9154|         7995|
+------------+--------+-------------+



## 2. Inferencia sin estado: una sola lectura basta (S10)

El modelo necesita el valor actual, cinco rezagos, la otra variable y la hora. En la v2 los rezagos salían de un búfer de 90 minutos en memoria. Ahora **vienen dentro del evento**, así que Spark pronostica cada fila por separado, sin guardar historial.

Prueba: se toma **un solo** mensaje del tópico y se pronostica con él, sin nada más.

In [15]:
def columna_feature(f, v):
    """Nombre de feature del 12b -> expresion sobre UN evento."""
    if f == v:
        return F.col("valor")
    if f.startswith(f"{v}_l"):
        return F.col("contexto.rezagos").getItem(f.split("_")[-1])
    if f == "delta_1h":
        return F.col("valor") - F.col("contexto.rezagos").getItem("l12")
    if f == "hora_sin":
        return F.sin((F.col("ts5") % 86400) / 3600.0 * 2 * math.pi / 24)
    if f == "hora_cos":
        return F.cos((F.col("ts5") % 86400) / 3600.0 * 2 * math.pi / 24)
    return F.col("contexto.simultaneas").getItem(f)   # la otra variable, en el mismo bloque de 5 min

def preparar(d, v):
    feats = META["variables"][v]["features"]
    x = (d.where(F.col("variable") == v)
          .withColumn("ts5", (F.floor(F.unix_timestamp("ts") / 300) * 300).cast("long"))
          .withColumn("ultimo", F.col("valor")))
    x = x.select("evento_id", "estacion_id", "variable", "ts", "ts5", "ultimo", "retraso_min", "origen",
                 "partition", "offset", *[columna_feature(f, v).alias(f) for f in feats if f != v],
                 F.col("valor").alias(v))
    return x.dropna(subset=feats)          # sin contexto completo no se pronostica

def inferir(d):
    """Sin estado: cada fila se transforma sola (VectorAssembler + LinearRegression)."""
    salidas = []
    for v, modelo in MODELOS.items():
        salidas.append(modelo.transform(preparar(d, v)).select(
            "evento_id", "estacion_id", "variable", "ts", "ts5", "ultimo",
            F.col("prediction").alias("pred"), "retraso_min", "origen", "partition", "offset"))
    out = salidas[0]
    for s in salidas[1:]:
        out = out.unionByName(s)
    return out

# Prueba: UN solo mensaje -> pronostico
# se elige un mensaje que traiga todo lo que su modelo pide (rezagos y la otra variable)
tiene_todo = F.lit(False)
for v in MODELOS:
    otras = [f for f in META["variables"][v]["features"] if f in VARS_CONOCIDAS and f != v]
    cond = F.col("variable") == v
    for o in otras:
        cond = cond & F.col("contexto.simultaneas").getItem(o).isNotNull()
    tiene_todo = tiene_todo | cond
uno = (mias.where((F.col("validacion") == "valido") & completo & tiene_todo)
           .orderBy(F.desc("ts")).limit(1))
print("Mensaje de entrada (uno solo):")
print(json.dumps(json.loads(uno.first()["json"]), indent=1, ensure_ascii=False))
print("\nPronostico a 1 hora calculado SOLO con ese mensaje:")
inferir(uno).select("estacion_id", "variable", "ts", "ultimo", F.round("pred", 3).alias("pronostico_1h")).show(truncate=False)

Mensaje de entrada (uno solo):


{
 "version_esquema": 2,
 "evento_id": "bd4556cb-0d02-4009-89a5-94c2e12abc72",
 "estacion_id": "USGS-02054550",
 "variable": "ph",
 "codigo_parametro": "00400",
 "valor": 8.3,
 "fecha_medicion_utc": "2026-11-06T14:25:00+00:00",
 "fecha_ingesta_utc": "2026-11-06T14:34:03.252237+00:00",
 "retraso_min": 9.1,
 "origen": "vivo",
 "fuente": "USGS",
 "contexto": {
  "rezagos": {
   "l1": 8.3,
   "l2": 8.3,
   "l3": 8.3,
   "l6": 8.3,
   "l12": 8.3
  },
  "simultaneas": {
   "oxigeno_mgl": 9.8,
   "turbidez_fnu": 1.0,
   "conductividad_uscm": 405.0,
   "temperatura_c": 15.0
  }
 }
}

Pronostico a 1 hora calculado SOLO con ese mensaje:


[Stage 140:=====>           (1 + 2) / 3][Stage 141:=====>           (1 + 2) / 3]

+---------------+------------+-------------------+------+-------------+
|estacion_id    |variable    |ts                 |ultimo|pronostico_1h|
+---------------+------------+-------------------+------+-------------+
|USGS-0164579522|turbidez_fnu|2026-11-06 09:25:00|13.5  |12.716       |
+---------------+------------+-------------------+------+-------------+



## 3. Riesgo y evento de negocio con contrato versionado (S6)

El riesgo es la probabilidad de que el valor dentro de 1 hora quede fuera del rango habitual de la estación (percentiles 5 y 95 del historial), suponiendo un error normal con sigma igual al RMSE de validación.

El evento de negocio lleva `version_esquema` y un `evento_id` **determinista** (estación, variable y bloque de 5 min): si por la semántica al menos una vez se publica dos veces, el consumidor lo reconoce como el mismo.

In [16]:
def ncdf(z): return 0.5 * (1 + math.erf(z / math.sqrt(2)))

def riesgo(est, v, pred):
    u = META["umbrales"].get(f"{est}|{v}")
    s = META["variables"][v]["sigma"]
    if not u or s <= 0: return None, None
    d = DIRECCION.get(v, "ambas")
    p_alta = 1 - ncdf((u["p95"] - pred) / s)
    p_baja = ncdf((u["p05"] - pred) / s)
    p = p_alta if d == "alta" else p_baja if d == "baja" else min(1.0, p_alta + p_baja)
    return p, u

def evento_negocio(r, p, u):
    ts_med = datetime.fromtimestamp(int(r["ts5"]), timezone.utc).isoformat()
    return {"version_esquema": 1,
            "evento_id": str(uuid.uuid5(uuid.NAMESPACE_URL, f"{TIPO_NEGOCIO}|{r['estacion_id']}|{r['variable']}|{int(r['ts5'])}")),
            "tipo": TIPO_NEGOCIO, "integrante": INTEGRANTE, "estacion_id": r["estacion_id"], "variable": r["variable"],
            "valor_actual": round(float(r["ultimo"]), 3), "pronostico_1h": round(float(r["pred"]), 3), "riesgo": round(p, 3),
            "umbral_p05": u["p05"], "umbral_p95": u["p95"], "direccion": DIRECCION.get(r["variable"]),
            "ts_medicion_utc": ts_med, "creado_utc": datetime.now(timezone.utc).isoformat(),
            "estado": "creada", "fuente": "USGS", "modelo": META["variables"][r["variable"]]["ganador"],
            "evento_origen_id": r["evento_id"]}

def publicar_negocio(ev):
    publicar_kafka([(f"{ev['estacion_id']}|{ev['variable']}", json.dumps(ev, ensure_ascii=False))], TOPIC_NEGOCIO)

## 4. Error de predicción medido en vivo (S11)

Cada pronóstico apunta a un instante una hora después. Cuando llega la lectura real de ese instante, se mide el error del **modelo** y el de la **persistencia** (haber repetido el último valor). Así se compara en producción lo mismo que se comparó en la validación batch del 12b.

Este registro de pronósticos pendientes es estado de **monitoreo**, no de inferencia: el pronóstico no lo usa, y se borra solo (máximo 2 horas).

In [17]:
PEND = {}   # (estacion, variable, ts5_objetivo) -> (pronostico, valor al pronosticar)
ERR = {v: {m: {"n": 0, "abs": 0.0, "sq": 0.0} for m in ("modelo", "persistencia")} for v in VARIABLES}
ERR_ULT = {}  # (estacion, variable) -> error absoluto del modelo en la ultima comparacion

def medir_error(pdf):
    """pdf: filas pronosticadas del lote. Primero registra, despues compara (una fila nunca se compara consigo misma)."""
    nuevas = 0
    for r in pdf.itertuples():
        PEND[(r.estacion_id, r.variable, int(r.ts5) + 3600)] = (float(r.pred), float(r.ultimo))
    for r in pdf.itertuples():
        k = (r.estacion_id, r.variable, int(r.ts5))
        if k in PEND:
            pred, ult = PEND.pop(k)
            real = float(r.ultimo)
            for m, e in (("modelo", real - pred), ("persistencia", real - ult)):
                a = ERR[r.variable][m]; a["n"] += 1; a["abs"] += abs(e); a["sq"] += e * e
            ERR_ULT[(r.estacion_id, r.variable)] = abs(real - pred)
            nuevas += 1
    if len(pdf):
        limite = int(pdf["ts5"].max()) - 7200
        for k in [k for k in PEND if k[2] < limite]:
            del PEND[k]
    return nuevas

def resumen_error():
    filas = []
    for v in VARIABLES:
        m = META["variables"][v]
        fila = {"variable": v, "RMSE_validacion_modelo": round(m["rmse"], 4),
                "RMSE_validacion_persistencia": round(m["rmse_persistencia"], 4)}
        for mod in ("modelo", "persistencia"):
            a = ERR[v][mod]
            fila[f"RMSE_vivo_{mod}"] = round(math.sqrt(a["sq"] / a["n"]), 4) if a["n"] else None
        fila["comparaciones_vivo"] = ERR[v]["modelo"]["n"]
        filas.append(fila)
    return pd.DataFrame(filas)

## 5. Métricas hacia Prometheus (S9 y S11)

Tres grupos en el Pushgateway, para que uno no borre al otro:

| Grupo (`job`) | Quién lo envía | Métricas |
|---|---|---|
| `u2_usgs_batch` | una vez, al iniciar | RMSE y MAE de validación del 12b (panel **batch estático**) |
| `u2_usgs` | cada lote (`foreachBatch`) | KPI de negocio, último valor, pronóstico, riesgo y **error en vivo** |
| `u2_usgs_stream` | el listener de Spark | **lag del consumidor**, eventos por segundo, duración del lote, filas de estado y duplicados descartados |

Pushgateway es un atajo: el job de streaming no expone un `/metrics` propio que Prometheus pueda leer. Por eso el tablero también muestra cuántos segundos pasaron desde el último envío: si el job muere, el Pushgateway sigue mostrando el último valor y ese panel se pone rojo.

In [18]:
def lbl(**k): return ",".join(f'{a}="{b}"' for a, b in k.items())

class Metricas:
    def __init__(self): self.L = []
    def tipo(self, n, ayuda, t="gauge"): self.L += [f"# HELP {n} {ayuda}", f"# TYPE {n} {t}"]
    def val(self, n, valor, **k): self.L.append(f"{n}{{{lbl(integrante=ID, **k)}}} {valor}")
    def enviar(self, job):
        req = urllib.request.Request(f"{PUSHGATEWAY}/metrics/job/{job}/instance/{ID}",
                                     data=("\n".join(self.L) + "\n").encode(), method="PUT")
        try:
            urllib.request.urlopen(req, timeout=5).read(); return True
        except Exception as e:
            print(f"  Pushgateway ({job}) no responde:", e); return False

def push_batch_estatico():
    m = Metricas()
    m.tipo("usgs_rmse_validacion", "RMSE de validacion del 12b (corte cronologico)")
    m.tipo("usgs_mae_validacion", "MAE de validacion del 12b")
    m.tipo("usgs_mejora_vs_persistencia_pct", "Mejora del RMSE frente a la persistencia (%)")
    for v in VARIABLES:
        x = META["variables"][v]
        m.val("usgs_rmse_validacion", x["rmse"], variable=v, modelo="modelo")
        m.val("usgs_rmse_validacion", x["rmse_persistencia"], variable=v, modelo="persistencia")
        m.val("usgs_mae_validacion", x["mae"], variable=v, modelo="modelo")
        m.val("usgs_mejora_vs_persistencia_pct", x["mejora_vs_persistencia_pct"], variable=v)
    return m.enviar("u2_usgs_batch")

TOT = {"lotes": 0, "alertas": 0, "pronosticos": 0}
ESTADO, ULT_ALERTA = {}, {}

def push_lote(retraso, dur):
    m = Metricas()
    m.tipo("usgs_lotes_total", "Lotes con pronosticos", "counter"); m.val("usgs_lotes_total", TOT["lotes"])
    m.tipo("usgs_pronosticos_total", "Pronosticos emitidos", "counter"); m.val("usgs_pronosticos_total", TOT["pronosticos"])
    m.tipo("usgs_alertas_total", "Eventos de negocio emitidos", "counter"); m.val("usgs_alertas_total", TOT["alertas"])
    m.tipo("usgs_retraso_min", "Retraso medio entre medicion y publicacion USGS (min, solo eventos vivos)"); m.val("usgs_retraso_min", f"{retraso:.2f}")
    m.tipo("usgs_duracion_lote_s", "Duracion del procesamiento del lote (s)"); m.val("usgs_duracion_lote_s", f"{dur:.3f}")
    m.tipo("usgs_ultimo_valor", "Ultima lectura"); m.tipo("usgs_pronostico_1h", "Pronostico a 1 hora"); m.tipo("usgs_riesgo", "Riesgo de salir del rango")
    for (est, v), s in ESTADO.items():
        m.val("usgs_ultimo_valor", s["ultimo"], estacion=est, variable=v)
        m.val("usgs_pronostico_1h", s["pred"], estacion=est, variable=v)
        if s["riesgo"] is not None: m.val("usgs_riesgo", s["riesgo"], estacion=est, variable=v)
    m.tipo("usgs_error_abs", "Error absoluto del ultimo pronostico ya comparado con la lectura real")
    for (est, v), e in ERR_ULT.items(): m.val("usgs_error_abs", e, estacion=est, variable=v)
    m.tipo("usgs_rmse_vivo", "RMSE medido en vivo"); m.tipo("usgs_mae_vivo", "MAE medido en vivo")
    m.tipo("usgs_comparaciones_vivo_total", "Pronosticos ya comparados con la lectura real", "counter")
    for v in VARIABLES:
        for mod, a in ERR[v].items():
            if a["n"]:
                m.val("usgs_rmse_vivo", f"{math.sqrt(a['sq'] / a['n']):.5f}", variable=v, modelo=mod)
                m.val("usgs_mae_vivo", f"{a['abs'] / a['n']:.5f}", variable=v, modelo=mod)
        m.val("usgs_comparaciones_vivo_total", ERR[v]["modelo"]["n"], variable=v)
    m.enviar("u2_usgs")

class Observador(StreamingQueryListener):
    """Lee el progreso de cada micro-lote de Spark y lo manda a Prometheus."""
    def __init__(self):
        self.ultimo = None; self.leidos = 0; self.dup = 0
    def onQueryStarted(self, e): pass
    def onQueryIdle(self, e): pass
    def onQueryTerminated(self, e): pass
    def onQueryProgress(self, e):
        p = e.progress
        if not p.name or not p.name.startswith(f"u2v3_{ID}"): return
        self.ultimo = p
        self.leidos += p.numInputRows
        src = p.sources[0]
        def suma(js):
            try: return sum(sum(int(o) for o in parts.values()) for parts in json.loads(js).values())
            except Exception: return None
        fin, ult = suma(src.endOffset), suma(getattr(src, "latestOffset", None))
        met = dict(src.metrics or {})
        if None not in (fin, ult):
            lag = ult - fin                      # total de mensajes sin procesar, sumando particiones
        else:
            lag = float(met.get("maxOffsetsBehindLatest", 0))
        filas_estado = sum(op.numRowsTotal for op in p.stateOperators)
        self.dup = sum(int(op.customMetrics.get("numDroppedDuplicateRows", 0)) for op in p.stateOperators)
        tardios = sum(op.numRowsDroppedByWatermark for op in p.stateOperators)
        m = Metricas()
        m.tipo("usgs_kafka_lag_mensajes", "Mensajes de Kafka que Spark aun no proceso (lag del consumidor)"); m.val("usgs_kafka_lag_mensajes", lag)
        m.tipo("usgs_eventos_total", "Mensajes leidos de Kafka", "counter"); m.val("usgs_eventos_total", self.leidos)
        m.tipo("usgs_entrada_eventos_s", "Eventos por segundo que llegan"); m.val("usgs_entrada_eventos_s", p.inputRowsPerSecond or 0)
        m.tipo("usgs_procesados_eventos_s", "Eventos por segundo que Spark procesa"); m.val("usgs_procesados_eventos_s", p.processedRowsPerSecond or 0)
        m.tipo("usgs_lote_ms", "Duracion total del micro-lote (ms)"); m.val("usgs_lote_ms", p.durationMs.get("triggerExecution", 0))
        m.tipo("usgs_estado_filas", "Filas guardadas en el estado de Spark (solo deduplicacion)"); m.val("usgs_estado_filas", filas_estado)
        m.tipo("usgs_duplicados_descartados", "Duplicados descartados en el ultimo lote"); m.val("usgs_duplicados_descartados", self.dup)
        m.tipo("usgs_tardios_descartados", "Filas descartadas por llegar despues del watermark (ultimo lote)"); m.val("usgs_tardios_descartados", tardios)
        m.tipo("usgs_batch_id", "Ultimo batchId"); m.val("usgs_batch_id", p.batchId)
        m.enviar("u2_usgs_stream")

if "OBS" in globals():
    spark.streams.removeListener(OBS)
OBS = Observador()
spark.streams.addListener(OBS)
print("Panel batch (validacion 12b) enviado:", push_batch_estatico())

Panel batch (validacion 12b) enviado: True


## 6. Streaming: Kafka → validación → deduplicación → pronóstico → negocio y métricas

1. Lee `agua-usgs-eventos-v2` con `maxOffsetsPerTrigger` (el lag baja de a poco).
2. Deja solo eventos válidos de mis variables.
3. **Deduplica** por (estación, variable, tiempo del evento) dentro del watermark: Kafka entrega *al menos una vez* y el productor puede reenviar.
4. En `foreachBatch`, por cada micro-lote: **pronostica fila por fila** (sin estado), calcula el riesgo, emite el evento de negocio, mide el error en vivo y envía las métricas.

In [19]:
def lote(df, bid):
    t0 = time.monotonic()
    pdf = inferir(df).toPandas()      # pronostico fila por fila sobre el micro-lote (sin estado)
    if pdf.empty:
        push_lote(0.0, time.monotonic() - t0); return
    pdf = pdf.sort_values("ts5")
    vivos = pdf[pdf["origen"] == "vivo"]["retraso_min"]
    retraso = float(vivos.mean()) if len(vivos) else 0.0
    TOT["lotes"] += 1; TOT["pronosticos"] += len(pdf)
    nuevas = 0
    for r in pdf.to_dict("records"):
        p, u = riesgo(r["estacion_id"], r["variable"], r["pred"])
        ESTADO[(r["estacion_id"], r["variable"])] = {"ultimo": r["ultimo"], "pred": r["pred"], "riesgo": p}
        if p is not None and p >= RIESGO_ALERTA:
            k = (r["estacion_id"], r["variable"])
            if r["ts5"] - ULT_ALERTA.get(k, 0) >= ENFRIAMIENTO_MIN * 60:
                publicar_negocio(evento_negocio(r, p, u)); ULT_ALERTA[k] = r["ts5"]
                TOT["alertas"] += 1; nuevas += 1
    comparadas = medir_error(pdf)
    dur = time.monotonic() - t0
    push_lote(retraso, dur)
    print(f"lote {bid}: {len(pdf)} pronosticos | {TOT['alertas']} eventos de negocio (+{nuevas}) | "
          f"{comparadas} comparados con la lectura real | {dur:.2f}s")

# Reinicio limpio del estado de la demostracion
ESTADO.clear(); ULT_ALERTA.clear(); PEND.clear(); ERR_ULT.clear()
for v in VARIABLES:
    for a in ERR[v].values(): a.update(n=0, abs=0.0, sq=0.0)
TOT.update(lotes=0, alertas=0, pronosticos=0); OBS.leidos = 0

CHK = f"{ARTIFACTS}/chk_negocio_v3_{ID}"
if LIMPIAR_CHECKPOINT and os.path.exists(CHK): shutil.rmtree(CHK)

entrada = parsear(fuente_stream("latest" if MODO == "vivo" else "earliest", MAX_POR_LOTE))
validos = entrada.where((F.col("validacion") == "valido") & F.col("variable").isin(VARIABLES))
sin_duplicados = (validos.withWatermark("ts", WATERMARK)
                         .dropDuplicatesWithinWatermark(["estacion_id", "variable", "ts"]))
# La inferencia se aplica dentro de foreachBatch (funcion lote): una sola lectura de Kafka y un solo estado de deduplicacion.
q = (sin_duplicados.writeStream.queryName(f"u2v3_{ID}").foreachBatch(lote)
     .option("checkpointLocation", CHK).trigger(processingTime=TRIGGER).start())
print(f"Corriendo {DURACION_MIN} min en modo {MODO}. Abre Grafana (localhost:43000) mientras tanto.")
q.awaitTermination(DURACION_MIN * 60)
q.stop()
print("Resumen:", TOT, "| mensajes leidos de Kafka:", OBS.leidos)

Corriendo 8 min en modo reproduccion. Abre Grafana (localhost:43000) mientras tanto.
lote 0: 306 pronosticos | 5 eventos de negocio (+5) | 0 comparados con la lectura real | 2.13s


lote 1: 305 pronosticos | 9 eventos de negocio (+4) | 299 comparados con la lectura real | 1.92s


lote 2: 305 pronosticos | 14 eventos de negocio (+5) | 305 comparados con la lectura real | 1.90s
lote 3: 304 pronosticos | 18 eventos de negocio (+4) | 304 comparados con la lectura real | 1.75s


lote 4: 300 pronosticos | 21 eventos de negocio (+3) | 300 comparados con la lectura real | 1.70s


lote 5: 303 pronosticos | 24 eventos de negocio (+3) | 298 comparados con la lectura real | 1.66s


lote 6: 125 pronosticos | 24 eventos de negocio (+0) | 65 comparados con la lectura real | 1.31s


... (12 outputs left out)

lote 40: 29 pronosticos | 40 eventos de negocio (+1) | 29 comparados con la lectura real | 1.81s


lote 41: 14 pronosticos | 41 eventos de negocio (+1) | 14 comparados con la lectura real | 1.38s


lote 42: 24 pronosticos | 41 eventos de negocio (+0) | 24 comparados con la lectura real | 1.31s


lote 43: 12 pronosticos | 43 eventos de negocio (+2) | 6 comparados con la lectura real | 1.50s
Resumen: {'lotes': 24, 'alertas': 43, 'pronosticos': 3055} | mensajes leidos de Kafka: 26330


## 7. Evidencias

### 7.1 La inferencia no guarda estado (S10)
En el progreso de Spark, el **único** operador con estado es la deduplicación (S8). El pronóstico no aparece como estado.

In [20]:
p = OBS.ultimo
if p is None:
    print("La consulta no llego a procesar ningun lote.")
else:
    print("Operadores con estado en la consulta:")
    for op in p.stateOperators:
        print(f"  {op.operatorName:<30} filas en estado={op.numRowsTotal}  memoria={op.memoryUsedBytes} B  "
              f"duplicados descartados (ultimo lote)={op.customMetrics.get('numDroppedDuplicateRows')}")
    print("\nLag y throughput del ultimo lote:")
    src = p.sources[0]
    print("  offsets procesados hasta :", src.endOffset)
    print("  ultimo offset en Kafka   :", getattr(src, "latestOffset", None))
    print("  metricas de la fuente    :", dict(src.metrics or {}))
    print(f"  eventos/s entrada={p.inputRowsPerSecond:.1f}  procesados={p.processedRowsPerSecond:.1f}  lote={p.durationMs.get('triggerExecution')} ms")

Operadores con estado en la consulta:
  dedupeWithinWatermark          filas en estado=1458  memoria=272648 B  duplicados descartados (ultimo lote)=96

Lag y throughput del ultimo lote:
  offsets procesados hasta : {"agua-usgs-eventos-v2":{"0":10559,"1":10164,"2":5607}}
  ultimo offset en Kafka   : {"agua-usgs-eventos-v2":{"0":17853,"1":17192,"2":9491}}
  metricas de la fuente    : {'minOffsetsBehindLatest': '3884', 'maxOffsetsBehindLatest': '7294', 'avgOffsetsBehindLatest': '6068.666666666667'}
  eventos/s entrada=59.9  procesados=289.5  lote=2068 ms


### 7.2 Riesgo por serie y umbral

In [21]:
rs = pd.DataFrame([{"estacion": e, "variable": v, "ultimo": s["ultimo"], "pronostico_1h": round(s["pred"], 3),
                    "riesgo": None if s["riesgo"] is None else round(s["riesgo"], 4)} for (e, v), s in ESTADO.items()])
if len(rs):
    rs = rs.sort_values("riesgo", ascending=False)
    print(rs.head(10).to_string(index=False))
    print("riesgo maximo:", rs["riesgo"].max(), "  umbral de alerta:", RIESGO_ALERTA)

       estacion     variable  ultimo  pronostico_1h  riesgo
USGS-0164578734 turbidez_fnu    52.3         52.463  1.0000
USGS-0164578734           ph     6.8          6.665  0.9997
  USGS-01589290           ph     7.3          7.289  0.9990
  USGS-01649500 turbidez_fnu    45.2         52.831  0.9986
USGS-0164579522 turbidez_fnu    50.0         46.590  0.9961
  USGS-02074556           ph     6.9          6.919  0.9959
USGS-0164579522           ph     6.5          6.636  0.7019
  USGS-01649500           ph     7.4          7.371  0.6642
  USGS-01649190           ph     7.3          7.310  0.4413
  USGS-01650800 turbidez_fnu     5.8          5.514  0.4138
riesgo maximo: 1.0   umbral de alerta: 0.5


### 7.3 Error en vivo frente a la validación batch (S11)
Misma métrica (RMSE), mismo rival (persistencia): arriba lo que dijo la validación del 12b, abajo lo que pasa en producción.

In [22]:
print(resumen_error().to_string(index=False))
print("\nPronosticos que aun esperan su lectura real:", len(PEND))

    variable  RMSE_validacion_modelo  RMSE_validacion_persistencia  RMSE_vivo_modelo  RMSE_vivo_persistencia  comparaciones_vivo
          ph                  0.0683                        0.0753            0.0647                  0.0683                1201
turbidez_fnu                  8.2011                        8.5651           18.3764                 18.6048                1206

Pronosticos que aun esperan su lectura real: 316


### 7.4 Evento de negocio en Kafka

In [23]:
neg = (fuente_batch(TOPIC_NEGOCIO)
       .select(F.col("key").cast("string").alias("clave"), F.col("value").cast("string").alias("evento"), "partition", "offset"))
print("eventos de negocio en el topico:", neg.count())
ult = neg.orderBy(F.desc("offset")).first()
if ult:
    print(json.dumps(json.loads(ult["evento"]), indent=1, ensure_ascii=False))

eventos de negocio en el topico: 74
{
 "version_esquema": 1,
 "evento_id": "90882b86-cd30-56c5-8045-6fb520753008",
 "tipo": "alerta_calidad",
 "integrante": "Jhenderson Machaca",
 "estacion_id": "USGS-01649500",
 "variable": "turbidez_fnu",
 "valor_actual": 45.2,
 "pronostico_1h": 52.831,
 "riesgo": 0.999,
 "umbral_p05": 1.5,
 "umbral_p95": 28.4,
 "direccion": "alta",
 "ts_medicion_utc": "2026-11-06T09:05:00+00:00",
 "creado_utc": "2026-11-06T14:45:31.763282+00:00",
 "estado": "creada",
 "fuente": "USGS",
 "modelo": "LinearRegression",
 "evento_origen_id": "1ee40ece-4626-45ac-9bf9-c7e00c4a3876"
}
